# 🗺️ AFETSONAR — Notebook 7: 7-Katmanlı İnteraktif Master Harita
### Plan v2 — Phase 7: Sahaya Çıkacak Karar Destek Arayüzü

**Amaç:** Phase 5'te ürettiğimiz tüm çıktıları (`priority_test.csv`, `road_graph.gpickle`, `voronoi.geojson`) tek bir **interaktif Folium haritasında** birleştirmek. AFAD/UMKE/AKUT ekiplerinin WhatsApp'tan açıp anında karar verebileceği, mobil uyumlu bir arayüz.

---

## 🎯 7 Katman

| # | Katman | İçerik | Renk |
|---|---|---|---|
| 1 | **Base tiles** | OSM + Satellite (toggle) | — |
| 2 | **Bina poligonları** | 60 bina, hasar rengine göre, popup'lı | yeşil/sarı/turuncu/kırmızı |
| 3 | **Hasar ısı haritası** | HeatMap (priority weighted) | gradient |
| 4 | **Voronoi sorumluluk** | Hastane bölgeleri, yarı şeffaf | mavi tonları |
| 5 | **Yol grafı** | Açık yollar yeşil, kapalı kırmızı | green/red |
| 6 | **A\* rotalar** | Hastane → top 3 öncelik, AntPath animasyonlu | mavi |
| 7 | **Helikopter LZ** | Açık alan önerileri, helikopter ikonu | mor |

**Çıktı:** `afetsonar_master_map.html` — tek dosya, internet bağlantısı bile gerekmez (tile'lar hariç), WhatsApp'tan paylaşılabilir.

---

## 1️⃣ Kurulum ve Path'ler

**Bu hücre ne yapıyor?** Drive mount + folium kurulum + Phase 5 çıktı klasörünü tanımlama.

**Beklenen çıktı:** `✅ Tüm Phase 5 dosyaları bulundu` mesajı.

**Hata olursa:** Phase 5 dosyaları yoksa → Notebook 5'i çalıştır, çıktıları `outputs/phase5/` altında oluştur.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip install -q folium networkx shapely 2>&1 | tail -1

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/AFETSONAR') if IN_COLAB else Path('./AFETSONAR')
PHASE5 = ROOT / 'outputs' / 'phase5'
OUT = ROOT / 'outputs' / 'phase7'
OUT.mkdir(parents=True, exist_ok=True)

required = ['priority_test.csv', 'road_graph.gpickle', 'voronoi.geojson']
missing = [f for f in required if not (PHASE5/f).exists()]
if missing:
    print(f"❌ Eksik dosya(lar): {missing}")
    print(f"   Beklenen konum: {PHASE5}")
else:
    print(f"✅ Tüm Phase 5 dosyaları bulundu: {PHASE5}")

Mounted at /content/drive
✅ Tüm Phase 5 dosyaları bulundu: /content/drive/MyDrive/AFETSONAR/outputs/phase5


## 2️⃣ Import'lar ve Phase 5 Verilerini Yükle

**Bu hücre ne yapıyor?** Tüm kütüphaneleri import edip Phase 5'in 3 çıktısını belleğe alıyor.

**Beklenen çıktı:** Bina sayısı, graf node/edge sayısı, Voronoi feature sayısı.

**Hata olursa:** `pickle` versiyonu uyumsuzluğu → Colab Python sürümünü kontrol et.

In [2]:
import json, pickle
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, AntPath, MarkerCluster, MiniMap, Fullscreen
import networkx as nx
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, Point

# Binalar
buildings = pd.read_csv(PHASE5 / 'priority_test.csv')
print(f"🏢 {len(buildings)} bina yüklendi")

# Yol grafı
with open(PHASE5 / 'road_graph.gpickle', 'rb') as f:
    G = pickle.load(f)
blocked_count = sum(1 for u,v,k,d in G.edges(keys=True,data=True) if d.get('blocked'))
print(f"🛣️  Graf: {G.number_of_nodes()} node, {G.number_of_edges()} edge ({blocked_count} kapalı)")

# Hastaneler/Voronoi
with open(PHASE5 / 'voronoi.geojson') as f:
    voronoi_geo = json.load(f)
hospitals = pd.DataFrame([{
    'name': feat['properties']['name'],
    'type': feat['properties']['type'],
    'lon': feat['geometry']['coordinates'][0],
    'lat': feat['geometry']['coordinates'][1],
} for feat in voronoi_geo['features']])
print(f"🏥 {len(hospitals)} hastane/toplanma noktası")

# Harita merkezi
CENTER = [buildings['lat'].mean(), buildings['lon'].mean()]
print(f"📍 Harita merkezi: {CENTER}")

🏢 60 bina yüklendi
🛣️  Graf: 235 node, 463 edge (4 kapalı)
🏥 4 hastane/toplanma noktası
📍 Harita merkezi: [np.float64(40.999734387927084), np.float64(28.979242502332607)]


## 3️⃣ Ana Harita ve Base Tile Katmanları (Katman 1)

**Bu hücre ne yapıyor?** Folium ana haritasını oluşturup 2 base tile ekliyor: standart OSM ve uydu (Esri WorldImagery).

**Beklenen çıktı:** Boş harita (sol üst LayerControl ile değiştirilebilir).

**Hata olursa:** Esri tile yüklenmezse internet sorunu → sadece OSM kullan.

In [3]:
m = folium.Map(location=CENTER, zoom_start=15, tiles=None, control_scale=True)

# Katman 1a: OSM standart
folium.TileLayer('OpenStreetMap', name='🗺️ OSM Standart', control=True).add_to(m)

# Katman 1b: Uydu görüntüsü
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri WorldImagery',
    name='🛰️ Uydu Görüntüsü',
    control=True
).add_to(m)

# Eklentiler
MiniMap(toggle_display=True).add_to(m)
Fullscreen().add_to(m)

print("✅ Ana harita + 2 base tile hazır")

✅ Ana harita + 2 base tile hazır


## 4️⃣ Katman 2 — Bina Poligonları (Hasar Renkli + Popup)

**Bu hücre ne yapıyor?** 60 binayı renkli marker olarak ekliyor. Her marker'a tıklayınca bina ID, hasar sınıfı, alan, popülasyon ve öncelik skoru gösteriliyor. AFAD ekibinin sahada en çok kullanacağı bilgi bu.

**Beklenen çıktı:** Her bina renkli daire, tıklanınca popup açılıyor.

**Renk kodu:** destroyed=kırmızı, major=turuncu, minor=sarı, no_damage=yeşil

**Hata olursa:** 60 marker fazla görünürse `MarkerCluster` ile gruplanır (zaten kullanıyoruz).

In [4]:
color_map = {1: '#2ecc71', 2: '#f1c40f', 3: '#e67e22', 4: '#e74c3c'}
icon_map  = {1: 'ok-sign', 2: 'info-sign', 3: 'warning-sign', 4: 'remove-sign'}

bldg_layer = folium.FeatureGroup(name='🏢 2. Binalar (hasar renkli)', show=True)
cluster = MarkerCluster(disable_clustering_at_zoom=16).add_to(bldg_layer)

for _, b in buildings.iterrows():
    popup_html = f"""
    <div style='font-family:sans-serif; min-width:200px'>
      <h4 style='margin:0; color:{color_map[b.damage_class]}'>{b.building_id}</h4>
      <hr style='margin:4px 0'>
      <b>Hasar:</b> {b.damage_name}<br>
      <b>Alan:</b> {b.area_m2:.0f} m²<br>
      <b>Tahmini kişi:</b> {b.pop_estimate:.0f}<br>
      <b>Öncelik skoru:</b> <span style='color:red'><b>{b.priority_score:.0f}</b></span><br>
      <b>Sıralama:</b> #{b.priority_rank}
    </div>
    """
    folium.CircleMarker(
        location=[b.lat, b.lon],
        radius=6 + b.severity_w * 0.4,
        color=color_map[b.damage_class],
        fill=True, fill_opacity=0.85, weight=2,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{b.building_id} • {b.damage_name}"
    ).add_to(cluster)

bldg_layer.add_to(m)
print("✅ Katman 2: 60 bina marker eklendi")

✅ Katman 2: 60 bina marker eklendi


## 5️⃣ Katman 3 — Hasar Isı Haritası (HeatMap)

**Bu hücre ne yapıyor?** `priority_score` ile ağırlıklandırılmış HeatMap çiziyor — yüksek öncelikli alanlar daha "sıcak" görünüyor. Komuta merkezi için hızlı durum farkındalığı.

**Beklenen çıktı:** Kırmızı/turuncu lekeler en kritik bölgelerde.

**Hata olursa:** Tüm puanlar düşükse harita soluk → no_damage'leri filtreliyoruz.

In [5]:
# no_damage'i çıkar, ağırlık olarak normalize priority kullan
heat_df = buildings[buildings['damage_class'] > 1].copy()
max_p = heat_df['priority_score'].max()
heat_data = [[r.lat, r.lon, r.priority_score / max_p] for _, r in heat_df.iterrows()]

heat_layer = folium.FeatureGroup(name='🔥 3. Hasar Isı Haritası', show=False)
HeatMap(
    heat_data, radius=25, blur=20, max_zoom=17,
    gradient={'0.2':'blue','0.4':'lime','0.6':'yellow','0.8':'orange','1.0':'red'}
).add_to(heat_layer)
heat_layer.add_to(m)
print(f"✅ Katman 3: HeatMap ({len(heat_data)} ağırlıklı nokta)")

✅ Katman 3: HeatMap (44 ağırlıklı nokta)


## 6️⃣ Katman 4 — Voronoi Sorumluluk Bölgeleri

**Bu hücre ne yapıyor?** Hastane noktalarından scipy ile Voronoi diyagramı üretip her hücreyi yarı şeffaf poligon olarak çiziyor. Hangi hastane hangi bölgeden sorumlu — triage atama mantığı.

**Beklenen çıktı:** 4 mavi tonlu poligon, hastane noktalarının etrafında.

**Hata olursa:** Sonsuza giden Voronoi hücreleri → bbox ile clipping yapıyoruz.

In [6]:
from shapely.geometry import box as shp_box

# Bbox sınırı (clip için)
margin = 0.005
clip_box = shp_box(
    buildings['lon'].min() - margin, buildings['lat'].min() - margin,
    buildings['lon'].max() + margin, buildings['lat'].max() + margin
)

# Shapely 1.8+ voronoi_diagram kullanıyoruz — sonsuz kenarları otomatik
# clip ediyor, scipy'nin -1 vertex problemi yok.
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint

voronoi_layer = folium.FeatureGroup(name='🔵 4. Voronoi Sorumluluk', show=False)
voronoi_colors = ['#3498db', '#9b59b6', '#1abc9c', '#34495e']

mp = MultiPoint([(r.lon, r.lat) for _, r in hospitals.iterrows()])
vd = voronoi_diagram(mp, envelope=clip_box)

# vd.geoms sırası mp ile aynı olmayabilir → her hücreyi içerdiği hastaneye eşle
for cell in vd.geoms:
    cell_clipped = cell.intersection(clip_box)
    if cell_clipped.is_empty:
        continue
    # Hangi hastane bu hücrenin içinde?
    owner_idx = None
    for i, (_, h) in enumerate(hospitals.iterrows()):
        if cell_clipped.contains(Point(h.lon, h.lat)):
            owner_idx = i
            break
    if owner_idx is None:
        continue

    if cell_clipped.geom_type == 'Polygon':
        coords = [[y, x] for x, y in cell_clipped.exterior.coords]
        folium.Polygon(
            coords, color=voronoi_colors[owner_idx % 4], fill=True,
            fill_color=voronoi_colors[owner_idx % 4], fill_opacity=0.20, weight=2,
            popup=f"Sorumlu: {hospitals.iloc[owner_idx]['name']}"
        ).add_to(voronoi_layer)

voronoi_layer.add_to(m)
print("✅ Katman 4: Voronoi sorumluluk bölgeleri")

✅ Katman 4: Voronoi sorumluluk bölgeleri


## 7️⃣ Katman 5 — OSM Yol Grafı (Açık/Kapalı)

**Bu hücre ne yapıyor?** Phase 5'te ürettiğimiz `road_graph.gpickle`'daki tüm edge'leri çiziyor. `blocked=True` olanlar kırmızı kalın, açıklar yeşil ince.

**Beklenen çıktı:** Tüm yol ağı, kapalılar dikkat çekici.

**Hata olursa:** Edge geometry'si yoksa (sadece u-v line) → node koordinatlarından düz çizgi çekiyoruz.

In [7]:
road_layer = folium.FeatureGroup(name='🛣️ 5. Yol Grafı (kapalı=kırmızı)', show=True)

for u, v, k, d in G.edges(keys=True, data=True):
    # Edge geometry varsa kullan, yoksa düz çizgi
    if 'geometry' in d:
        coords = [[y, x] for x, y in d['geometry'].coords]
    else:
        coords = [[G.nodes[u]['y'], G.nodes[u]['x']],
                  [G.nodes[v]['y'], G.nodes[v]['x']]]
    blocked = d.get('blocked', False)
    folium.PolyLine(
        coords,
        color='#c0392b' if blocked else '#27ae60',
        weight=4 if blocked else 1.5,
        opacity=0.85 if blocked else 0.5,
        popup='🚧 KAPALI' if blocked else None
    ).add_to(road_layer)

road_layer.add_to(m)
print(f"✅ Katman 5: {G.number_of_edges()} edge ({blocked_count} kapalı)")

✅ Katman 5: 463 edge (4 kapalı)


## 8️⃣ Katman 6 — A* Rotalar (Top 3 Öncelik, AntPath)

**Bu hücre ne yapıyor?** En yüksek öncelikli 3 binayı seçip her biri için en yakın hastaneden A* rotası hesaplıyor. Folium AntPath ile **animasyonlu mavi çizgi** olarak çiziyor — yön anlaşılır olsun diye.

**Beklenen çıktı:** 3 hareketli rota, başlangıç-bitiş popup'lı.

**Hata olursa:** Bir hedef için kara rota yoksa o atlanır, Cell 9'da helikopter LZ önerilir.

In [8]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2-lat1); dl = np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def nearest_node(G, lat, lon):
    best, best_d = None, float('inf')
    for n, nd in G.nodes(data=True):
        d = (nd['y']-lat)**2 + (nd['x']-lon)**2
        if d < best_d:
            best, best_d = n, d
    return best

route_layer = folium.FeatureGroup(name='🚑 6. A* Rotalar (top 3)', show=True)
top3 = buildings.nlargest(3, 'priority_score')
no_route_targets = []

for _, target in top3.iterrows():
    # En yakın hastane
    hospitals['d'] = hospitals.apply(
        lambda r: haversine(r.lat, r.lon, target.lat, target.lon), axis=1)
    nearest = hospitals.sort_values('d').iloc[0]

    orig = nearest_node(G, nearest.lat, nearest.lon)
    dest = nearest_node(G, target.lat, target.lon)

    try:
        path = nx.astar_path(G, orig, dest, weight='weight')
        coords = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in path]
        length_m = sum(G[u][v][0].get('length', 0) for u,v in zip(path[:-1], path[1:]))

        AntPath(
            coords, color='#2980b9', weight=5, delay=800,
            dash_array=[10,20], pulse_color='#fff',
            popup=f"<b>{nearest['name']} → {target.building_id}</b><br>"
                  f"Hedef: {target.damage_name}<br>"
                  f"Mesafe: {length_m:.0f}m<br>"
                  f"Öncelik: {target.priority_score:.0f}"
        ).add_to(route_layer)
        print(f"  ✅ {nearest['name']} → {target.building_id}: {length_m:.0f}m")
    except nx.NetworkXNoPath:
        no_route_targets.append(target)
        print(f"  ❌ {target.building_id}: kara rota yok → LZ fallback")

route_layer.add_to(m)
print(f"✅ Katman 6: {len(top3)-len(no_route_targets)} rota çizildi")

  ✅ Toplanma-D → B0006: 1897m
  ✅ Toplanma-B → B0046: 1404m
  ✅ Toplanma-B → B0037: 0m
✅ Katman 6: 3 rota çizildi


## 9️⃣ Katman 7 — Hastaneler + Helikopter LZ Önerileri

**Bu hücre ne yapıyor?**
1. 4 hastane/toplanma noktasını + ikon ekliyor.
2. Cell 8'de kara rota bulunamayan hedefler için **helikopter LZ** öneriyor (Voronoi vertex'leri = doğal açık alan adayları, prototip).

**Beklenen çıktı:** Mavi hastane ikonları + (varsa) mor helikopter ikonları.

**Hata olursa:** `no_route_targets` boşsa LZ katmanı boş kalır, normal.

In [9]:
hosp_layer = folium.FeatureGroup(name='🏥 7a. Hastane/Toplanma', show=True)
for _, h in hospitals.iterrows():
    icon_color = 'blue' if h['type'] == 'hospital' else 'darkgreen'
    icon_name  = 'plus' if h['type'] == 'hospital' else 'home'
    folium.Marker(
        [h.lat, h.lon],
        popup=f"<b>{h['name']}</b><br>Tip: {h['type']}",
        tooltip=h['name'],
        icon=folium.Icon(color=icon_color, icon=icon_name, prefix='fa')
    ).add_to(hosp_layer)
hosp_layer.add_to(m)

# Helikopter LZ — kara rota yoksa
lz_layer = folium.FeatureGroup(name='🚁 7b. Helikopter LZ', show=True)
if no_route_targets:
    for target in no_route_targets:
        # Prototip: hedefe en yakın Voronoi vertex'i (gerçekte park polygonu olur)
        best_v, best_d = None, float('inf')
        for v in vor.vertices:
            d = haversine(target.lat, target.lon, v[1], v[0])
            if 0 < d < best_d:
                best_v, best_d = v, d
        if best_v is not None:
            folium.Marker(
                [best_v[1], best_v[0]],
                popup=f"<b>🚁 LZ önerisi</b><br>Hedef: {target.building_id}<br>Mesafe: {best_d:.0f}m",
                tooltip=f"LZ → {target.building_id}",
                icon=folium.Icon(color='purple', icon='plane', prefix='fa')
            ).add_to(lz_layer)
            # LZ → hedef hava hattı
            folium.PolyLine(
                [[best_v[1], best_v[0]], [target.lat, target.lon]],
                color='purple', weight=3, opacity=0.7, dash_array='10,10'
            ).add_to(lz_layer)
else:
    print("ℹ️  Tüm top 3 hedef için kara rota mevcut, LZ gerekmedi")

lz_layer.add_to(m)
print("✅ Katman 7: Hastaneler + LZ önerileri")

ℹ️  Tüm top 3 hedef için kara rota mevcut, LZ gerekmedi
✅ Katman 7: Hastaneler + LZ önerileri


## 🔟 Lejant + LayerControl + Kaydet

**Bu hücre ne yapıyor?** Sağ üste LayerControl (katman aç/kapat) ve sol alta HTML lejant ekliyor, sonra haritayı `afetsonar_master_map.html` olarak kaydediyor.

**Beklenen çıktı:** Tek HTML dosyası (~200-500KB), çift tıkla aç → tarayıcıda interaktif harita.

**Hata olursa:** Drive yazma izni → remount.

In [10]:
# Lejant
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 12px 16px; border: 2px solid #333;
            border-radius: 8px; font-family: sans-serif; font-size: 13px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3);">
  <b style="font-size:14px">🚨 AFETSONAR Lejant</b><hr style="margin:6px 0">
  <span style="color:#e74c3c">●</span> Yıkık (destroyed)<br>
  <span style="color:#e67e22">●</span> Ağır hasar (major)<br>
  <span style="color:#f1c40f">●</span> Hafif hasar (minor)<br>
  <span style="color:#2ecc71">●</span> Hasarsız<br>
  <hr style="margin:6px 0">
  <span style="color:#c0392b">━</span> Kapalı yol<br>
  <span style="color:#2980b9">━</span> A* rotası<br>
  <span style="color:purple">┄</span> Helikopter hattı
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Başlık
title_html = '''
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background: rgba(231,76,60,0.95); color: white;
            padding: 8px 20px; border-radius: 6px; font-family: sans-serif;
            font-weight: bold; font-size: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">
  🚨 AFETSONAR — Karar Destek Haritası
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# LayerControl en sona
folium.LayerControl(collapsed=False, position='topright').add_to(m)

# Kaydet
out_path = OUT / 'afetsonar_master_map.html'
m.save(str(out_path))
print(f"💾 KAYDEDİLDİ: {out_path}")
print(f"   Boyut: {out_path.stat().st_size/1024:.1f} KB")
m

💾 KAYDEDİLDİ: /content/drive/MyDrive/AFETSONAR/outputs/phase7/afetsonar_master_map.html
   Boyut: 385.5 KB


---
## ✅ Phase 7 Tamamlandı!

### Üretilen Çıktı
**`afetsonar_master_map.html`** — Tek dosya, paylaşılabilir, mobil uyumlu.

### Saha Kullanım Senaryosu
1. **Komuta merkezi** Phase 5 → Phase 7 pipeline'ını çalıştırır (~5 dk)
2. HTML dosyasını WhatsApp grubuna düşer
3. **AFAD ekip lideri** mobilde açar:
   - Önce **HeatMap** ile büyük resmi görür
   - **Bina katmanını** açıp en yüksek öncelikli noktalara dokunur
   - **A\* rotasını** takip ederek hareket eder
   - Yol kapalıysa **Helikopter LZ** koordinatına yönlendirilir
4. **Voronoi** ile hangi ekibin hangi bölgeden sorumlu olduğu net

### Plan v2 İlerleme
- ✅ Phase 1, 2, 4, 5, **7**
- ⏳ Phase 2 v4 eğitimi (paralel)
- ⏳ Phase 8: Drone telemetry simulator + canlı triage akışı

### Sonraki Adım: Gerçek Veriye Geçiş
Phase 2 v4 (`teacher_v4_best_ema.pth`, mIoU_no_bg ≥ 0.55) bittiğinde:
1. `predict.py` ile gerçek test split üzerinde inference
2. Çıktıyı Notebook 5 Cell 3'e besle (sentetik DataFrame yerine)
3. Notebook 7 otomatik olarak gerçek veriyle çalışır — değişiklik gerekmez!

🚨 **AFETSONAR — "Sadece nerede değil, NASIL ulaşılır da söylüyoruz. Ulaşılamıyorsa ALTERNATİF sunuyoruz."**